<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/sarvam_12L.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://huggingface.co/frankmorales2020/sarvam-30b-topo-governed-cbp-multi

In [1]:
!pip install bitsandbytes -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 62.2 MB/s eta 0:00:00


In [2]:
!nvidia-smi

Mon Oct 19 12:31:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   71C    P8             18W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## case1

In [ ]:
!pip install -q "transformers==4.57.2"
# then: Runtime -> Restart session, then run the cell below

In [2]:
!pip show transformers bitsandbytes

Name: transformers
Version: 4.57.2
Summary: State-of-the-art Machine Learning for JAX, PyTorch and TensorFlow
Home-page: https://github.com/huggingface/transformers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/transformers/graphs/contributors)
Author-email: transformers@huggingface.co
License: Apache 2.0 License
Location: /usr/local/lib/python3.13/dist-packages
Requires: filelock, huggingface-hub, numpy, packaging, pyyaml, regex, requests, safetensors, tokenizers, tqdm
Required-by: peft, sentence-transformers
---
Name: bitsandbytes
Version: 0.50.2
Summary: k-bit optimizers and matrix multiplication routines.
Home-page: https://github.com/bitsandbytes-foundation/bitsandbytes
Author: 
Author-email: Tim Dettmers <dettmers@cs.washington.edu>
License: 
Location: /usr/local/lib/python3.13/dist-packages
Requires: numpy, packaging, torch
Required-by: 


In [3]:
import os; os.environ["HF_HUB_DISABLE_XET"] = "1"   # plain HTTPS download, avoids 429 rate limits

from google.colab import userdata
from huggingface_hub import HfApi, login
import torch
import torch.nn as nn
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import gc
import hashlib
import time
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*AttentionMaskConverter.*")

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN)

# =====================================================================
# 1. TASK-AWARE MODEL ARCHITECTURE (4-TASK SUPPORT)
# =====================================================================
class TaskAwareModel(nn.Module):
    def __init__(self, base_model: nn.Module, hidden_size: int = 4096, device: torch.device = torch.device('cuda')):
        super().__init__()
        self.base_model = base_model
        self.hidden_size = hidden_size
        self.device = device

        self.classifier_A = nn.Linear(hidden_size, 2, dtype=torch.bfloat16).to(self.device)
        self.classifier_B = nn.Linear(hidden_size, 2, dtype=torch.bfloat16).to(self.device)
        self.classifier_C = nn.Linear(hidden_size, 2, dtype=torch.bfloat16).to(self.device)
        self.classifier_D = nn.Linear(hidden_size, 2, dtype=torch.bfloat16).to(self.device)
        self.current_task = 'A'

    def forward(self, input_ids, attention_mask=None):
        outputs = self.base_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
            return_dict=True,
        )

        hidden_states = outputs.hidden_states[-1]

        if attention_mask is not None:
            seq_lens = torch.eq(attention_mask, 1).int().sum(-1) - 1
            batch_idx = torch.arange(input_ids.shape[0], device=input_ids.device)
            last_hidden = hidden_states[batch_idx, seq_lens, :]
        else:
            last_hidden = hidden_states[:, -1, :]

        head = getattr(self, f'classifier_{self.current_task}')
        logits = head(last_hidden.to(head.weight.device))
        return logits

    def switch_task(self, task: str):
        assert task in ('A', 'B', 'C', 'D')
        self.current_task = task

# =====================================================================
# 2. ROBUST DOWNLOAD & VALIDATION PIPELINE FROM HUGGING FACE
# =====================================================================
if __name__ == "__main__":
    # Sarvam-30B's custom model code only computes correctly on this version
    assert transformers.__version__ == "4.57.2", \
        f"Need transformers 4.57.2, have {transformers.__version__}"

    MODEL_ID = 'sarvamai/sarvam-30b'
    REPO_ID = 'frankmorales2020/sarvam-30b-topo-governed-cbp-multi'
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    print("=" * 70)
    print("🚀 DOWNLOADING & INITIALIZING CERTIFIED MODEL FROM HUGGING FACE")
    print("=" * 70)

    print("\n[1/4] Downloading certified checkpoint from Hub (with exponential backoff)...")
    api = HfApi()

    max_retries = 6
    downloaded_file = None
    for attempt in range(max_retries):
        try:
            downloaded_file = api.hf_hub_download(
                repo_id=REPO_ID,
                filename="pytorch_model.bin"
            )
            break
        except Exception as e:
            wait_time = 30 * (attempt + 1)
            print(f"⚠️ Download interrupted (Attempt {attempt+1}/{max_retries}): {e}")
            if attempt == max_retries - 1:
                raise e
            print(f"⏳ Backing off for {wait_time} seconds to clear rate limit...")
            time.sleep(wait_time)

    print("[2/4] Loading tokenizer & configuring base model with BitsAndBytes 4-bit...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Reuse the model if it is already loaded in this session (skips the ~9 min load)
    if globals().get("base_model") is None:
        gc.collect(); torch.cuda.empty_cache()

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
            llm_int8_enable_fp32_cpu_offload=True,   # required for "lm_head": "cpu"
        )

        base_model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            quantization_config=bnb_config,
            device_map={"model": 0, "lm_head": "cpu"},     # backbone on GPU, 2 GB output layer on CPU
            dtype=torch.bfloat16,
            low_cpu_mem_usage=True,
            trust_remote_code=True,
        )
        base_model.config.use_cache = False

    print("[3/4] Instantiating TaskAwareModel and loading downloaded weights...")
    model = TaskAwareModel(base_model, hidden_size=4096, device=DEVICE)

    checkpoint = torch.load(downloaded_file, map_location="cpu")
    # Load ONLY the classifier heads; the full state dict overwrites and corrupts the 4-bit base
    head_state = {k: v for k, v in checkpoint["model_state_dict"].items()
                  if k.startswith("classifier_")}
    model.load_state_dict(head_state, strict=False)

    # ---------------- Integrity checks ----------------
    checks = {}

    info = api.get_paths_info(REPO_ID, ["pytorch_model.bin"])[0]
    expected_sha = info.lfs.sha256 if info.lfs else None
    sha = hashlib.sha256()
    with open(downloaded_file, "rb") as f:
        for chunk in iter(lambda: f.read(64 * 1024 * 1024), b""):
            sha.update(chunk)
    checks["Checkpoint SHA-256 matches Hub"] = expected_sha is not None and sha.hexdigest() == expected_sha

    expected_keys = {f"classifier_{t}.{p}" for t in "ABCD" for p in ("weight", "bias")}
    checks["All 8 head tensors present"] = set(head_state) == expected_keys
    checks["Head shapes match model"] = all(
        getattr(model, k.split(".")[0]).get_parameter(k.split(".")[1]).shape == v.shape
        for k, v in head_state.items())
    checks["Head weights finite"] = all(torch.isfinite(v.float()).all().item() for v in head_state.values())

    integrity_ok = all(checks.values())

    print("\n[Artifact Verification Check]")
    print(f" - Recorded Bedrock Hash: {checkpoint.get('state_hash', 'N/A')}  (stored value, not re-computed)")
    print(f" - Active Tasks Registry: {checkpoint.get('active_tasks', [])}")
    print(f" - Recorded Max Drift   : {checkpoint.get('max_drift', 0.0):.12f}  (stored value)")
    for name, ok in checks.items():
        print(f" - {name:32s}: {'PASS' if ok else 'FAIL'}")
    print(f" - Integrity Status     : {'PASS' if integrity_ok else 'FAIL'} "
          f"({sum(checks.values())}/{len(checks)} checks passed)")
    del checkpoint

    model.eval()

    print("\n[4/4] Executing Multi-Task Heads & 12-Language Validation Suite...")
    print("-" * 70)

    test_cases = [
        ('A', "Wall Street stocks advanced following strong quarterly corporate earnings reports."),
        ('B', "Astronomers detect unexpected water vapor signatures in a distant exoplanet atmosphere."),
        ('C', "The championship match concluded with a dramatic penalty shootout in extra time."),
        ('D', "Un film absolument magnifique avec des acteurs touchants et une mise en scène brillante."),
        ('A', "भारत की संस्कृति, विविधता और भाषाएं इसकी सबसे बड़ी ताकत हैं। (Foundational Hindi Check)")
    ]

    print("\n--- Running Task-Aware Classification Heads ---")
    with torch.no_grad():
        for task_id, text in test_cases:
            active_task = 'A' if 'Hindi' in text else task_id
            model.switch_task(active_task)
            inputs = tokenizer(text, return_tensors="pt", padding="max_length", truncation=True,
                               max_length=128, return_token_type_ids=False).to(DEVICE)
            logits = model(inputs["input_ids"], attention_mask=inputs["attention_mask"])
            probs = torch.softmax(logits.float(), dim=-1)
            pred_class = torch.argmax(probs, dim=-1).item()
            conf = probs[0][pred_class].item()

            print(f"[Task Head '{active_task}']")
            print(f" Input Text : \"{text}\"")
            print(f" Prediction : Class {pred_class} (Confidence: {conf:.4f})")
            print(f" Status     : {'Integrity checks PASS' if integrity_ok else 'Integrity checks FAILED'}\n")

    twelve_language_prompts = {
        "English": "Explain the concept of topological continual learning in one sentence:",
        "French": "Expliquez le concept de l'apprentissage continu topologique en une phrase :",
        "Hindi": "टोपोलॉजिकल कंटीन्यूअल लर्निंग की अवधारणा को एक वाक्य में समझाएं:",
        "Bengali": "টপোলজিক্যাল কন্টিনিউয়াল লার্নিং-এর ধারণাটি এক বাক্যে ব্যাখ্যা করুন:",
        "Marathi": "टोपॉलॉजिकल कन्टिन्युअल लर्निंगची संकल्पना एका वाक्यात स्पष्ट करा:",
        "Telugu": "టోపోలాజికల్ కంటిన్యూయల్ లెర్నింగ్ భావనను ఒక వాక్యంలో వివరించండి:",
        "Tamil": "டோபோலாஜிக்கல் கன்டினுவல் லேர்னிங் கருத்தை ஒரே வாக்கியத்தில் விளக்கவும்:",
        "Gujarati": "ટોપોલોજિકલ કન્ટિન્યુઅલ લર્નિંગના ખ્યાલને એક વાક્યમાં સમજાવો:",
        "Kannada": "ಟೋಪೋಲಾಜಿಕಲ್ ಕಂಟಿನ್ಯೂಯಲ್ ಲರ್ನಿಂಗ್ ಪರಿಕಲ್ಪನೆಯನ್ನು ಒಂದೇ ವಾಕ್ಯದಲ್ಲಿ ವಿವರಿಸಿ:",
        "Malayalam": "ടോപ്പോളജിക്കൽ കണ്ടിന്യുവൽ ലേണിംഗ് (topological continual learning) എന്ന ആശയം ഒറ്റ വാചകത്തിൽ വിശദീകരിക്കുക:",
        "Punjabi": "ਟੋਪੋਲਾਜੀਕਲ ਨਿਰੰਤਰ ਸਿੱਖਣ ਦੀ ਧਾਰਨਾ ਨੂੰ ਇੱਕ ਵਾਕ ਵਿੱਚ ਸਮਝਾਓ:",
        "Odia": "ଟୋପୋଲୋଜିକାଲ୍ କଣ୍ଟିନ୍ୟୁଆଲ୍ ଲର୍ଣ୍ଣିଂର ଧାରଣାକୁ ଗୋଟିଏ ବାକ୍ୟରେ ବର୍ଣ୍ଣନା କରନ୍ତୁ:"
    }

    SYSTEM_PROMPT = (
        "You are an expert in machine learning. "
        "Topological continual learning is a continual-learning approach in which a neural network "
        "learns a sequence of new tasks without catastrophically forgetting earlier ones, by preserving "
        "the topological structure (shape and connectivity) of its learned internal representations "
        "while it adapts to new data. "
        "Answer the user's question in exactly ONE sentence, written entirely in the same language "
        "as the question, using only that language's script (technical terms may be added in English "
        "in parentheses). Do not add explanations, lists, or a second sentence."
    )

    print("\n--- Running 12-Language Generation Validation Suite ---")
    bos = tokenizer.bos_token_id
    with torch.no_grad():
        for lang, prompt in twelve_language_prompts.items():
            text = tokenizer.apply_chat_template(
                [{"role": "system", "content": SYSTEM_PROMPT},
                 {"role": "user", "content": prompt}],
                tokenize=False, add_generation_prompt=True, enable_thinking=False)
            if "</think>" not in text:                                 # empty think block: answer directly
                text = text + "<think>\n\n</think>\n\n"
            input_ids = tokenizer(text, return_tensors="pt", add_special_tokens=False,
                                  return_token_type_ids=False)["input_ids"]
            if bos is not None and input_ids[0, 0].item() != bos:      # model needs BOS token
                input_ids = torch.cat([torch.tensor([[bos]]), input_ids], dim=1)
            input_ids = input_ids.to(DEVICE)
            outputs = model.base_model.generate(
                input_ids=input_ids,
                attention_mask=torch.ones_like(input_ids),
                max_new_tokens=200,
                do_sample=False,                                       # greedy: same answer every run
                repetition_penalty=1.0,
                pad_token_id=tokenizer.eos_token_id
            )
            response = tokenizer.decode(outputs[0][input_ids.shape[1]:], skip_special_tokens=True)
            if "</think>" in response:
                response = response.split("</think>", 1)[1]
            response = response.split("---")[0].strip().split("\n\n")[0]
            print(f"\n[{lang}]")
            print(f"Prompt : {prompt}")
            print(f"Answer : {response.strip()}")

    print("\n" + "=" * 70)
    print("✅ HUB-BASED INFERENCE, INTEGRITY, AND 12-LANGUAGE TESTS COMPLETED.")
    print("=" * 70)

🚀 DOWNLOADING & INITIALIZING CERTIFIED MODEL FROM HUGGING FACE

[1/4] Downloading certified checkpoint from Hub (with exponential backoff)...
[2/4] Loading tokenizer & configuring base model with BitsAndBytes 4-bit...


Loading checkpoint shards:   0%|          | 0/26 [00:00<?, ?it/s]

[3/4] Instantiating TaskAwareModel and loading downloaded weights...

[Artifact Verification Check]
 - Recorded Bedrock Hash: f0e8a92d0309c0662dd7216256ed3ff929f021d37c6ccd600afb5b4a194d9cde  (stored value, not re-computed)
 - Active Tasks Registry: ['A', 'B', 'C', 'D']
 - Recorded Max Drift   : 0.000000000000  (stored value)
 - Checkpoint SHA-256 matches Hub  : PASS
 - All 8 head tensors present      : PASS
 - Head shapes match model         : PASS
 - Head weights finite             : PASS
 - Integrity Status     : PASS (4/4 checks passed)

[4/4] Executing Multi-Task Heads & 12-Language Validation Suite...
----------------------------------------------------------------------

--- Running Task-Aware Classification Heads ---
[Task Head 'A']
 Input Text : "Wall Street stocks advanced following strong quarterly corporate earnings reports."
 Prediction : Class 0 (Confidence: 0.5898)
 Status     : Integrity checks PASS

[Task Head 'B']
 Input Text : "Astronomers detect unexpected water vap